# E-AMPHI — high *effective* amplitude, and the fidelity ceiling

A2 showed nominal 12× delivers only **3.48× effective** amplitude, because K̂_B^E1 is an estimate
(NCC with E2 = 0.246) and most injected energy lands off the true fingerprint direction.

**Reaching 12× effective would need nominal ≈ 41, where PSNR falls to ~34 dB — visible damage.**
That ceiling is not a nuisance; it is a result:

> Fingerprint-estimation quality bounds how much *true* device signal any injection-based
> verification scheme can deliver at a fixed perceptual cost. At NCC(E1,E2) = 0.246, ~75% of the
> injected energy is wasted as visible noise.

Directly relevant to SIREN-style coating verification, which assumes an injected pattern can be
made arbitrarily strong at imperceptible cost.

### Design
| α nominal | PSNR | ≈ effective | run? |
|---|---|---|---|
| 12 | 44.4 dB | 3.5× | already have |
| **18** | **40.9 dB** | **≈6.3×** | **yes** |
| **24** | **38.4 dB** | **≈8.3×** | **yes** |
| 41 | 33.7 dB | ≈13.9× | **no** — visible |

Two adapters, ~50 min training, ~40 min generation, ~10 min measurement. Fresh `OUT_DIR` because
`ALPHAS` is inside the hashed config. Reuses the base fingerprints and the paired seed bank.

In [ ]:
# %% H-CFG  (mirrors E_AMP; only ALPHAS and OUT change)
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
import csv as _csv
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"
BASE=os.path.join(DRIVE,"inv_channel/E_INV_P0_v3")
AMP =os.path.join(DRIVE,"inv_channel/E_AMP")
ROOT=os.path.join(DRIVE,"inv_channel/E_AMPHI")
for d in ["","train_png","adapters","gens","csv","meta","fingerprints"]:
    os.makedirs(os.path.join(ROOT,d),exist_ok=True)
CACHE="/content/hi_cache"; os.makedirs(CACHE,exist_ok=True)

@dataclass
class CFG:
    STAGES: tuple = ("H0",)
    MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    CROP: int = 1024
    ALPHAS: tuple = (18.0, 24.0)          # nominal; ~6.3x and ~8.3x effective
    N_T: int = 50
    SEED: int = 0
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    G_PER: int = 500; GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    N_PERM: int = 10000; ALPHA: float = 0.01
    GRAD_CKPT: bool = False
C=CFG(); MEAS=C.CROP
CFG_SHA=hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k!="STAGES"},
                                  sort_keys=True,default=str).encode()).hexdigest()
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False
subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
try:
    import peft.import_utils as _piu; _piu.is_torchao_available=lambda: False
    import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available=lambda: False
except Exception: pass
Image.MAX_IMAGE_PIXELS=None
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)
def all_tags(): return [f"amp_{str(a).replace('.','p')}" for a in C.ALPHAS]
def alpha_of(t): return dict(zip(all_tags(),C.ALPHAS))[t]
def tdir(t): return os.path.join(ROOT,"train_png",t)
def adir(t): return os.path.join(ROOT,"adapters",t)
def gdir(t): return os.path.join(ROOT,"gens",t)
MAN={}
for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
    MAN.setdefault((r["role"],r["split"]),[]).append(r["path"])
BANK=["A","B","C","D","D2"]
def K_amp(r): return np.load(os.path.join(AMP,"fingerprints",f"K_{r}.npy"))
KB1=np.load(os.path.join(BASE,"fingerprints","K_B_E1.npy"))
print(f"protocol SHA {CFG_SHA[:16]} | alphas {C.ALPHAS} | device={DEV} dtype={DT}")

In [ ]:
# %% forensic core (identical to the base study)
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def load_rgb(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        return np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); return (r/float(r.std())).astype(np.float32)
def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    return float((a*b).sum()/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
def inject(rgb,K,a): return np.clip(rgb*(1.0+a*K[...,None]),0,255).astype(np.float32)

In [ ]:
# %% H0 — materialise, and MEASURE the fidelity ceiling (this is the result, not a side-check)
def h0():
    hdr=["alpha","idx","rho_KB","rho_KA","psnr","clip_frac"]
    p=os.path.join(ROOT,"csv","h0_real_dose.csv")
    KB2=K_amp("B"); KA2=K_amp("A")
    if not os.path.exists(p):
        for a in (0.0,)+tuple(C.ALPHAS):
            for i,fp in enumerate(MAN[("A","H")]):
                rgb=load_rgb(fp)
                raw=rgb*(1.0+a*KB1[...,None]) if a>0 else rgb
                cf=float(((raw>255)|(raw<0)).mean()); img=np.clip(raw,0,255).astype(np.float32)
                d=img-rgb; ps=float(20*np.log10(255/max(np.sqrt((d**2).mean()),1e-9))) if a>0 else 99.0
                Y=(0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]).astype(np.float32)
                W=wres(Y)
                append_row(p,hdr,[a,i,f"{ncc(W,Y*KB2):.6e}",f"{ncc(W,Y*KA2):.6e}",
                                  f"{ps:.2f}",f"{cf:.6f}"])
            print(f"[H0] real dose alpha={a} done")
    d=pd.read_csv(p); g=d.groupby("alpha")
    base=float(g["rho_KB"].median().loc[0.0])
    R_nat_B=3.14047e-02
    print(f"\n{'alpha':>7s} {'rho->K_B':>11s} {'alpha_eff':>10s} {'PSNR dB':>9s} {'clip %':>8s} "
          f"{'rho->K_A':>11s}")
    rows=[]
    for a,sub in g:
        eff=(sub["rho_KB"].median()-base)/R_nat_B
        rows.append((a,float(eff),float(sub["psnr"].median()),float(sub["clip_frac"].median()*100)))
        print(f"{a:7.1f} {sub['rho_KB'].median():11.4e} {eff:10.2f} "
              f"{sub['psnr'].median():9.2f} {sub['clip_frac'].median()*100:8.3f} "
              f"{sub['rho_KA'].median():11.4e}")
    ajson({"rows":rows,"R_natural_B":R_nat_B},os.path.join(ROOT,"h0_fidelity.json"),indent=2)
    print("\n  THE CEILING IS THE RESULT: report alpha_effective against PSNR, and state that")
    print("  reaching 12x effective needs nominal ~41 at ~34 dB, i.e. visible damage.")
    for t in all_tags():
        o=tdir(t); os.makedirs(o,exist_ok=True)
        if len(glob.glob(os.path.join(o,"*.png")))>=C.N_T: continue
        for i,fp in enumerate(MAN[("A","T")]):
            Image.fromarray(inject(load_rgb(fp),KB1,alpha_of(t)).astype(np.uint8)).save(
                os.path.join(o,f"{i:04d}.png"),compress_level=1)
        print(f"[H0] materialised {t} (alpha={alpha_of(t)})")
if stage("H0"): h0()

In [ ]:
# %% H1 — train one adapter per amplitude
def _expected(tag):
    return {"tag":tag,"alpha":alpha_of(tag),"rank":C.RANK,"steps":C.STEPS,"meas":MEAS,
            "seed":C.SEED,"config_sha":CFG_SHA,"model":C.MODEL,"dtype":str(DT)}
def adapter_is_valid(tag):
    p=os.path.join(adir(tag),"train_meta.json")
    if not os.path.exists(os.path.join(adir(tag),"pytorch_lora_weights.safetensors")): return False
    m=json.load(open(p)) if os.path.exists(p) else None
    return m is not None and all(m.get(k)==v for k,v in _expected(tag).items()) and m.get("effective",False)
def _prompt_embeds():
    from diffusers import StableDiffusion3Pipeline
    p=os.path.join(ROOT,"adapters","_prompt.pt")
    if os.path.exists(p):
        z=torch.load(p,map_location=DEV); return z["pe"].to(DEV),z["pp"].to(DEV)
    pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
    with torch.no_grad():
        pe,_,pp,_=pipe.encode_prompt(prompt=C.CAPTION,prompt_2=C.CAPTION,prompt_3=C.CAPTION,
                                     device=DEV,num_images_per_prompt=1,
                                     do_classifier_free_guidance=False)
    torch.save({"pe":pe.cpu(),"pp":pp.cpu()},p); del pipe
    gc.collect(); torch.cuda.empty_cache()
    return pe.to(DEV),pp.to(DEV)

def train_arm(tag,pe,pp):
    if adapter_is_valid(tag): print(f"[H1] skip valid {tag}"); return
    if DEV!="cuda": HALT("H1 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    out=adir(tag)
    if os.path.exists(out): shutil.move(out,out+f"__stale_{int(time.time())}")
    torch.manual_seed(C.SEED); np.random.seed(C.SEED); random.seed(C.SEED)
    try:
        pipe=StableDiffusion3Pipeline.from_pretrained(
            C.MODEL,torch_dtype=DT,token=True,text_encoder=None,text_encoder_2=None,
            text_encoder_3=None,tokenizer=None,tokenizer_2=None,tokenizer_3=None).to(DEV)
    except Exception:
        pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
        pipe.text_encoder=pipe.text_encoder_2=pipe.text_encoder_3=None
    sch,vae,tr=pipe.scheduler,pipe.vae,pipe.transformer
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.RANK,lora_alpha=C.RANK,init_lora_weights="gaussian",
                              target_modules=list(C.TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    ps=[p for p in tr.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(ps,lr=C.LR,weight_decay=1e-4)
    lrs=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(C.STEPS/C.GRAD_ACC))
    opt.zero_grad(set_to_none=True)
    files=sorted(glob.glob(os.path.join(tdir(tag),"*.png")))
    if len(files)!=C.N_T: HALT(f"{tag}: {len(files)} training PNGs, expected {C.N_T}")
    cache=torch.stack([torch.from_numpy(
        np.asarray(Image.open(f).convert("RGB"),np.float32).copy()/127.5-1.0).permute(2,0,1)
        for f in files])
    sf=float(vae.config.scaling_factor); sh=float(vae.config.shift_factor or 0.0)
    ts,sg=sch.timesteps.to(DEV),sch.sigmas.to(DEV); nts=int(sch.config.num_train_timesteps)
    if len(sg)!=nts or len(ts)!=nts: HALT("scheduler array length mismatch — pin diffusers")
    g=torch.Generator().manual_seed(C.SEED); losses=[]; t0=time.time()
    print(f"\n[H1] {tag}: alpha={alpha_of(tag)}, rank {C.RANK}, {C.STEPS} steps")
    for st in range(C.STEPS):
        idx=torch.randint(0,cache.shape[0],(C.BATCH,),generator=g)
        px=cache[idx].to(DEV,dtype=DT)
        with torch.no_grad():
            lat=vae.encode(px).latent_dist.sample(); lat=((lat-sh)*sf).to(DT)
        noise=torch.randn_like(lat)
        u=torch.sigmoid(torch.randn(lat.shape[0],device=DEV,dtype=torch.float32))
        ii=(u*nts).long().clamp_(0,nts-1); sig=sg[ii].to(lat.dtype).view(-1,1,1,1)
        pred=tr(hidden_states=((1.0-sig)*lat+sig*noise).to(DT),timestep=ts[ii],
                encoder_hidden_states=pe.repeat(lat.shape[0],1,1).to(DT),
                pooled_projections=pp.repeat(lat.shape[0],1).to(DT),return_dict=False)[0]
        loss=tF.mse_loss(pred.float(),(noise-lat).float())/C.GRAD_ACC
        loss.backward()
        if (st+1)%C.GRAD_ACC==0:
            torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); lrs.step()
            opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach())*C.GRAD_ACC)
        if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
        if (st+1)%400==0:
            el=time.time()-t0
            print(f"    {st+1}/{C.STEPS} loss {np.mean(losses[-400:]):.5f} "
                  f"eta {(el/(st+1))*(C.STEPS-st-1)/60:.1f}min")
    sd=get_peft_model_state_dict(tr)
    bn=float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
    if not (bn>1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
        HALT(f"{tag}: DEAD adapter (||lora_B||={bn:.3e})")
    os.makedirs(out,exist_ok=True)
    StableDiffusion3Pipeline.save_lora_weights(save_directory=out,transformer_lora_layers=sd)
    m=_expected(tag); m.update({"loss_tail":float(np.mean(losses[-100:])),"lora_B_norm":bn,
                                "effective":True,"minutes":(time.time()-t0)/60})
    ajson(m,os.path.join(out,"train_meta.json"),indent=2)
    print(f"[H1] saved {tag}: ||lora_B||={bn:.2f} tail={m['loss_tail']:.5f} "
          f"({m['minutes']:.1f} min)")
    del pipe,tr,vae,cache; gc.collect(); torch.cuda.empty_cache()

if stage("H1"):
    pe,pp=_prompt_embeds()
    for i,t in enumerate(all_tags(),1):
        print(f"[H1] adapter {i}/{len(all_tags())}: {t}"); train_arm(t,pe,pp)
    bad=[t for t in all_tags() if not adapter_is_valid(t)]
    if bad: HALT(f"H1 incomplete: {bad}")
    print(f"[H1] PASS — {len(all_tags())} adapters")

In [ ]:
# %% H2 — generation, same paired seed bank as every other experiment
def h2():
    if DEV!="cuda": HALT("H2 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    bank=[(C.CAPTION,C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    pipe=StableDiffusion3Pipeline.from_pretrained(C.MODEL,torch_dtype=DT,token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True); prev=False
    try:
        for tag in all_tags():
            d_=gdir(tag); os.makedirs(d_,exist_ok=True)
            for t_ in glob.glob(os.path.join(d_,"*.tmp")): os.remove(t_)
            miss=[i for i in range(C.G_PER)
                  if not os.path.exists(os.path.join(d_,f"{i:05d}.png"))]
            if not miss: print(f"[H2] skip complete {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev=False
            pipe.load_lora_weights(adir(tag)); prev=True
            print(f"[H2] {tag}: {len(miss)} missing"); t0=time.time()
            for s in range(0,len(miss),C.GEN_BATCH):
                ix=miss[s:s+C.GEN_BATCH]
                gs=[torch.Generator(device=DEV).manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs=pipe(prompt=[bank[i][0] for i in ix],num_inference_steps=C.GEN_STEPS,
                              guidance_scale=C.CFG_SCALE,height=MEAS,width=MEAS,
                              generator=gs).images
                for i,im in zip(ix,imgs):
                    t_=os.path.join(d_,f"{i:05d}.png.tmp")
                    im.save(t_,format="PNG"); os.replace(t_,os.path.join(d_,f"{i:05d}.png"))
                if (s//C.GEN_BATCH)%25==0: print(f"    {s+len(ix)}/{len(miss)}")
            print(f"[H2] complete {tag} ({(time.time()-t0)/60:.0f} min)")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[H2] PASS")
if stage("H2"): h2()

In [ ]:
# %% H3 — measurement against the same 5-fingerprint bank
def h3():
    K=torch.stack([torch.from_numpy(K_amp(r)) for r in BANK]).to(_MD)
    p=os.path.join(ROOT,"csv","h3_measure.csv"); done=done_keys(p,["tag","gen_idx","K"])
    ex=ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            paths=[os.path.join(gdir(tag),f"{i:05d}.png") for i in range(C.G_PER)]
            todo=[i for i in range(C.G_PER) if not all((tag,str(i),r) in done for r in BANK)]
            if not todo: print(f"[H3] {tag}: complete"); continue
            fut={i:ex.submit(load_lum,paths[i],False) for i in todo[:6]}; t0=time.time()
            for pos,i in enumerate(todo):
                Z_np=fut.pop(i).result(); nx=pos+6
                if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum,paths[todo[nx]],False)
                W=torch.from_numpy(wres(Z_np)).to(_MD); Z=torch.from_numpy(Z_np).to(_MD)
                v=nccb(W,Z.unsqueeze(0)*K).cpu().numpy()
                for j,r in enumerate(BANK):
                    if (tag,str(i),r) in done: continue
                    append_row(p,["tag","gen_idx","K","rho"],[tag,i,r,f"{v[j]:.6e}"])
            print(f"[H3] {tag}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    finally:
        ex.shutdown(wait=True); gc.collect()
        if DEV=="cuda": torch.cuda.empty_cache()
    n=len(done_keys(p,["tag","gen_idx","K"])); exp=len(all_tags())*C.G_PER*len(BANK)
    if n!=exp: HALT(f"H3 incomplete: {n}/{exp}")
    print(f"[H3] PASS — {n}/{exp} rows")
if stage("H3"): h3()

In [ ]:
# %% H4 — the dose-response extended, plotted on EFFECTIVE amplitude
def h4():
    rng=np.random.default_rng(0)
    d=pd.read_csv(os.path.join(ROOT,"csv","h3_measure.csv"))
    fid=json.load(open(os.path.join(ROOT,"h0_fidelity.json")))
    eff={r[0]:r[1] for r in fid["rows"]}; psnr={r[0]:r[2] for r in fid["rows"]}
    def v(tag,K): return d[(d.tag==tag)&(d.K==K)].sort_values("gen_idx")["rho"].to_numpy(float)
    def sf(x,n=C.N_PERM):
        o=x.mean(); c=sum((x*rng.choice((-1.,1.),len(x))).mean()>=o for _ in range(n))
        return (c+1)/(n+1),float(o)
    # the E_AMP arms already measured, for one continuous curve
    PRIOR={1.0:-1.520e-05,1.6:-2.862e-05,3.0:-6.868e-05,6.0:-3.233e-05,12.0:-2.101e-05}
    out={"config_sha":CFG_SHA,"levels":{}}
    print(f"{'alpha_nom':>10s} {'alpha_eff':>10s} {'PSNR':>7s} {'contrast':>12s} {'t':>7s} {'p':>9s}")
    for a_prev,c_prev in sorted(PRIOR.items()):
        print(f"{a_prev:10.1f} {'(E_AMP)':>10s} {'':>7s} {c_prev:+12.3e} {'':>7s} {'':>9s}")
    for tag in all_tags():
        a=alpha_of(tag); ctr=v(tag,"B")-v(tag,"A")
        p_,o_=sf(ctr); t_=o_/(ctr.std(ddof=1)/np.sqrt(len(ctr)))
        out["levels"][a]={"alpha_effective":eff.get(a),"psnr":psnr.get(a),
                          "contrast":o_,"t":float(t_),"p":p_,"n":int(len(ctr))}
        print(f"{a:10.1f} {eff.get(a,float('nan')):10.2f} {psnr.get(a,float('nan')):7.1f} "
              f"{o_:+12.3e} {t_:7.2f} {p_:9.4g}{'  <== DETECTED' if p_<C.ALPHA else ''}")
    det=[a for a,r in out["levels"].items() if r["p"]<C.ALPHA]
    hi=max(eff.get(a,0) for a in out["levels"])
    print("\n"+"="*72)
    if det:
        print(f"TRANSFER DETECTED at nominal {min(det):g} "
              f"(~{eff.get(min(det)):.1f}x effective) — this changes the headline.")
    else:
        print(f"NO TRANSFER up to ~{hi:.1f}x EFFECTIVE amplitude (nominal {max(out['levels']):g}),")
        print(f"extending the previous ceiling of ~3.5x. Fidelity at that level: "
              f"{psnr.get(max(out['levels'])):.1f} dB.")
        print("Report alongside the fidelity ceiling: reaching 12x effective needs nominal ~41 at")
        print("~34 dB, i.e. VISIBLE damage. That bounds ANY injection-based verification scheme")
        print("using an estimated fingerprint, not just this one.")
    print("="*72)
    ajson(out,os.path.join(ROOT,"E_AMPHI_results.json"),indent=2)
if stage("H4"): h4()

## Running it
Set `C.STAGES` and re-run the config cell each time.

| session | STAGES | wall | check |
|---|---|---|---|
| 1 | `("H0",)` | ~10 min | the **fidelity table** — α_effective against PSNR |
| 2 | `("H1",)` | ~50 min | both ‖lora_B‖ inside the base range 69.5–73.1 |
| 3 | `("H2",)` | ~40 min | 1000 generations |
| 4 | `("H3","H4")` | ~10 min | **the extended dose-response on effective amplitude** |

H4 prints the E_AMP levels alongside the new ones so the curve is continuous from nominal 1 to 24.